In [ ]:
### Import libraries ###
# from google.colab import files
import matplotlib.pyplot as plt
import matplotlib

#set default plotting fonts
font = {'family' : 'sans-serif',
        'weight' : 'normal',
        'size'   : 20}

matplotlib.rc('font', **font)

import numpy as np
import os
import pandas as pd
import random
from sklearn import metrics
from sklearn.utils import shuffle

#data preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

#garbage collection (for saving RAM during training)
import gc


## Loading the Dataset

This dataset is a modified version of the PhysioNet/CinC Challenge data, which were contributed by the Massachusetts General Hospital’s Computational Clinical Neurophysiology Laboratory, and the Clinical Data Animation Laboratory.
***
**Class labels:**
- 0 = Arousal
- 1 = NREM1
- 2 = NREM2
- 3 = NREM3
- 4 = REM
***
**Class descriptions:**

<img src="https://github.com/BeaverWorksMedlytics2020/Data_Public/blob/master/Images/Week2/sleepStagesTable.svg?raw=true">

***
**Physiological signal description:**

O2-M1 - posterior brain activity (electroencephalography)

E1-M2 - left eye activity (electrooculography)

Chin1-Chin2 - chin movement (electromyography)

ABD - abdominal movement (electromyography)

CHEST - chest movement (electromyography)

AIRFLOW - respiratory airflow

ECG - cardiac activity (electrocardiography)
***
Run both cell blocks to get the challenge data.

In [ ]:
# Clone repo and move into data directory (only run this once)
if not os.path.exists("./Data_Public/ChallengeProjects/Week2/"):
    os.system("git clone https://github.com/BeaverWorksMedlytics2020/Data_Public")
os.chdir("./Data_Public/ChallengeProjects/Week2/")


## Loading Data in Memory
Run the cell below to extract the raw training and test data. It may take a minute or two to run through. Here are the variables containing the data you will get:

* **data_train**: np array shape (4000, 12000, 7). Contains 4000 samples (60s each) of 12000 data points (200Hz x 60s), for 7 different signals.
* **labels_train**: np array shape (4000,). Contains ground truth labels for data_train. The order of the labels corresponds to the order of the training data.
* **ID_train**: list of 4000 unique IDs. The order of the IDs corresponds to the order of the training data.
* **data_test**: np array shape (1000, 12000, 7). Contains 1000 samples (60s each) of 12000 data points (200Hz x 60s), for 7 different signals.
* **ID_test**: list of 1000 unique IDs. The order of the IDs corresponds to the order of the training data.

We encourage you to print each of these variables to see what they look like.

In [ ]:
### Run once to import data ###

def get_file_locs():
    '''
    find all files in 'training' and 'test' directories and put their names
    under 'training' and 'test' keys in the file_dict dictionary
    '''

    file_dict = {'training':[], 'test':[]}
    for data_type in file_dict:
        for file in os.listdir('./' + data_type):
            file_dict[data_type].append(data_type + '/' + file)

    return file_dict

def get_sample_data(data_type, id_number):
    '''
    get signal data, label, and filename associated with given data type and index num

    parameters:

     data_type -- Dictates whether sample comes from training set or test set.
                 This input must be either 'training' or 'test' (defaults to 'training')

     id_number -- Which sample ID should be returned? Must be 0-3999 if data_type is 'training'
                 or 0-999 if data_type is 'test' (defaults to random integer from 0-999)

    returns:

     sample_data -- dataframe with 1 row and 2 columns-- column "Signal" contains a series object
                    and column "Label" contains numeric label for that sample
    '''
    file = './' + data_type + '/' + str(id_number) + '.xz'

    #sample_data is a dataframe with 1 row and 2 columns--
    #"Signal" (contains a series object) and "Label" (contains numeric label)
    sample_data = pd.read_pickle('./' + file)

    return sample_data, file.split('/')[2]

file_dict = get_file_locs()
print(f"{len(file_dict['training'])} training samples found, {len(file_dict['test'])} test samples found")

data_train = np.zeros((4000, 12000, 7))
labels_train = np.zeros(4000)
ID_train = []
for i in range(4000):
  sample_data, ID = get_sample_data('training', i)
  data_train[i] = np.array(list(sample_data['Signal']), dtype=float).reshape(12000, 7)
  labels_train[i] = np.array(list(sample_data['Label']), dtype=float)
  ID_train.append(ID)
  if(i%500==0):
    print('Loading training sample ' + str(i))

data_test = np.zeros((1000, 12000, 7))
ID_test = []
for i in range(1000):
  sample_data, ID = get_sample_data('test', i)
  data_test[i] = np.array(list(sample_data['Signal']), dtype=float).reshape(12000, 7)
  ID_test.append(ID)
  if(i%500==0):
    print('Loading test sample ' + str(i))

4000 training samples found, 1000 test samples found
Loading training sample 0


/var/folders/2v/9k8fc_p92r57l4dlnm8g58m00000gn/T/ipykernel_96514/2173094696.py:50: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  labels_train[i] = np.array(list(sample_data['Label']), dtype=float)
/var/folders/2v/9k8fc_p92r57l4dlnm8g58m00000gn/T/ipykernel_96514/2173094696.py:50: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  labels_train[i] = np.array(list(sample_data['Label']), dtype=float)
/var/folders/2v/9k8fc_p92r57l4dlnm8g58m00000gn/T/ipykernel_96514/2173094696.py:50: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before 

Loading training sample 500


/var/folders/2v/9k8fc_p92r57l4dlnm8g58m00000gn/T/ipykernel_96514/2173094696.py:50: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  labels_train[i] = np.array(list(sample_data['Label']), dtype=float)
/var/folders/2v/9k8fc_p92r57l4dlnm8g58m00000gn/T/ipykernel_96514/2173094696.py:50: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  labels_train[i] = np.array(list(sample_data['Label']), dtype=float)
/var/folders/2v/9k8fc_p92r57l4dlnm8g58m00000gn/T/ipykernel_96514/2173094696.py:50: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before 

Loading training sample 1000


...

Loading training sample 3000


/var/folders/2v/9k8fc_p92r57l4dlnm8g58m00000gn/T/ipykernel_96514/2173094696.py:50: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  labels_train[i] = np.array(list(sample_data['Label']), dtype=float)
/var/folders/2v/9k8fc_p92r57l4dlnm8g58m00000gn/T/ipykernel_96514/2173094696.py:50: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  labels_train[i] = np.array(list(sample_data['Label']), dtype=float)
/var/folders/2v/9k8fc_p92r57l4dlnm8g58m00000gn/T/ipykernel_96514/2173094696.py:50: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before 

Loading training sample 3500


/var/folders/2v/9k8fc_p92r57l4dlnm8g58m00000gn/T/ipykernel_96514/2173094696.py:50: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  labels_train[i] = np.array(list(sample_data['Label']), dtype=float)
/var/folders/2v/9k8fc_p92r57l4dlnm8g58m00000gn/T/ipykernel_96514/2173094696.py:50: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  labels_train[i] = np.array(list(sample_data['Label']), dtype=float)
/var/folders/2v/9k8fc_p92r57l4dlnm8g58m00000gn/T/ipykernel_96514/2173094696.py:50: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before 

Loading test sample 0
Loading test sample 500


## Processing labels

In [ ]:
### Create label array for all training samples using categorical datatype ###
train_labels = np.ndarray(shape = (1, 4000))

#set labels to integers first
for i in range(4000):
    train_labels[0][i] = i//800 # This is a way to label each entry (since classes are in order)


# Feature Extraction

In [ ]:
selected_signals = [0,1,2,5,6]
rate = 200

def extract_features(X):
  features=[]
  for sample in X:
    sample_features=[]
    for idx in selected_signals:
      sig = sample[:, idx]

      fft_vals= np.abs(np.fft.rfft(sig))
      freqs = np.fft.rfftfreq(len(sig), d=1/rate)
      bands = {
        "delta": (0.5, 4),
        "theta": (4, 8),
        "alpha": (8, 13),
        "sigma": (11, 16),
        "beta": (13, 30),
        "gamma": (30, 64)
      }

      for low, high in bands.values():
        sample_features.append(np.sum(fft_vals[(freqs >= low) & (freqs < high)]))

      sample_features += [np.max(np.abs(sig)), np.std(sig)]

    features.append(sample_features)
  return np.array(features)

F_train   = extract_features(data_train)
F_test  = extract_features(data_test)

print(f"Feature shape: {F_train.shape, F_test.shape}")

print("finished")

Feature shape: ((4000, 40), (1000, 40))
finished


# Downsampling

In [ ]:
from scipy.signal import decimate

def downsample(X, factor=4):        # Taking 1 in 4 samples to train faster and hopefully help stop overfitting
    return decimate(X, factor, axis=1, ftype='fir', zero_phase=True).astype('float32')

data_train = downsample(data_train)
data_test = downsample(data_test)

## Shuffle and Partition

In [ ]:
### Shuffle and partition all train data

#(Training data is ordered by default so shuffling before partitioning is important)

#--Shuffle data_train--
#(Note that data is only shuffled in first dimension, which is what we want)
shuffled_idx = np.random.permutation(len(data_train))

data_train = data_train[shuffled_idx]
F_train = F_train[shuffled_idx]

train_labels = train_labels.squeeze(axis=0)
train_labels = train_labels[shuffled_idx]
#--Scale all labeled data in data_train--

#initialize standard scaler
scaler = StandardScaler()

#Standard scaler is meant for 2D arrays, so we reshape, scale, and then reshape again
reshaped_X_train = data_train.reshape((data_train.shape[0]*data_train.shape[1], data_train.shape[2])).copy()
reshaped_X_train = scaler.fit_transform(reshaped_X_train)
data_train = reshaped_X_train.reshape((data_train.shape[0], data_train.shape[1], data_train.shape[2]))

del reshaped_X_train #get rid of large temporary variable

#--Scale unlabeled test data--
#Because we scaled labeled data before training, we need to also scale test data --

#Standard scaler is meant for 2D arrays, so we reshape, apply scaling, and then
#reshape again to get back to original
reshaped_X_test = data_test.reshape((data_test.shape[0]*data_test.shape[1], data_test.shape[2])).copy()
reshaped_X_test = scaler.transform(reshaped_X_test)
data_test = reshaped_X_test.reshape((data_test.shape[0], data_test.shape[1], data_test.shape[2]))

del reshaped_X_test #get rid of large temporary variable

In [ ]:
#--create 3 partitions of provided training data--
# Note we are breaking up provided labeled data into training, validation, and "mock test" sets

val_size = 500
mocktest_size = 0 # For now made this 0 to get more training data lol, since we have separate test data

X_test = data_train[0:mocktest_size, :, :]
F_test_mock = F_train[0:mocktest_size]
y_test = train_labels[0:mocktest_size]

X_val = data_train[mocktest_size:mocktest_size+val_size, :, :]
F_val = F_train[mocktest_size:mocktest_size+val_size]
y_val = train_labels[mocktest_size:mocktest_size+val_size]

X_tr = data_train[mocktest_size+val_size:,:,:]
F_tr = F_train[mocktest_size+val_size:]
y_tr = train_labels[mocktest_size+val_size:]

del data_train
del train_labels



In [ ]:
# Make extracted features log scale and feature scale
F_tr = np.log10(F_tr + 1e-12)
F_val = np.log10(F_val + 1e-12)
# F_test_mock = np.log10(F_test_mock + 1e-12)
F_test = np.log10(F_test + 1e-12)

extracted_scaler = StandardScaler()
F_tr = extracted_scaler.fit_transform(F_tr)
F_val = extracted_scaler.transform(F_val)
# F_test_mock = extracted_scaler.transform(F_test_mock)
F_test = extracted_scaler.transform(F_test)

In [ ]:
# print(X_tr[0, :])
print(y_tr[0])

3.0


### Implement your model here

Feel free to use whatever architecture you want so long as it incorporates convolutions.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Added this to try to get more out of the data and help overfitting because that was a big problem - Liam
class SignalAugment(layers.Layer):
    def call(self, x, training=None):
        if not training:
            # Only augment during training
            return x
        x = x + tf.random.normal(tf.shape(x), stddev=0.05)
        x = x * tf.random.uniform((tf.shape(x)[0],1,1), 0.8, 1.2)
        return x

def conv_block(x, filters, k, pool, drop=0.1):
    x = layers.Conv1D(filters, k, padding="same")(x)
    x = layers.BatchNormalization()(x) # To keep scale of each layer more consistent
    x = layers.ReLU()(x)

    x = layers.Conv1D(filters, k, padding="same")(x) # 2 conv layers per block to help learn better patterns
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    x = layers.MaxPooling1D(pool)(x) # Keep only strongest responses
    x = layers.Dropout(drop)(x) # Help with generalization
    return x

signal_input = keras.Input(shape=X_tr.shape[1:], name="signal")
feature_input = keras.Input(shape=F_train.shape[1:], name="extracted")

x = SignalAugment()(signal_input)
x = conv_block(x, filters=16, k=25, pool=4) # Each block more filters and smaller k size to find more features and reduce time steps
x = conv_block(x, filters=32, k=15, pool=4)
x = conv_block(x, filters=64, k=9, pool=4)
x = conv_block(x, filters=64, k=5, pool=4)
x = layers.Concatenate()([layers.GlobalAveragePooling1D()(x), layers.GlobalMaxPooling1D()(x)]) # Take strongest and average signals

f = layers.Dense(64, activation='relu')(feature_input) # Learning from extracted features too
f = layers.Dense(32, activation='relu')(f)

both = layers.Concatenate()([x, f]) # Combine both signal layers and extracted feature layers
both = layers.Dropout(0.4)(both) # Dropout to stop ovefitting at the end
output = layers.Dense(5)(both)

model = keras.Model([signal_input, feature_input], output)
model.summary()

Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ signal (InputLayer) │ (None, 3000, 7)   │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ signal_augment_4    │ (None, 3000, 7)   │          0 │ signal[0][0]      │
│ (SignalAugment)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_32 (Conv1D)  │ (None, 3000, 16)  │      2,816 │ signal_augment_4… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 3000, 16)  │         64 │ conv1d_32[0][0]   │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_32 (ReLU)     │ (None, 3000, 16)  │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_33 (Conv1D)  │ (None, 3000, 16)  │      6,416 │ re_lu_32[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 3000, 16)  │         64 │ conv1d_33[0][0]   │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_33 (ReLU)     │ (None, 3000, 16)  │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_16    │ (None, 750, 16)   │          0 │ re_lu_33[0][0]    │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_20          │ (None, 750, 16)   │          0 │ max_pooling1d_16… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_34 (Conv1D)  │ (None, 750, 32)   │      7,712 │ dropout_20[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 750, 32)   │        128 │ conv1d_34[0][0]   │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_34 (ReLU)     │ (None, 750, 32)   │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_35 (Conv1D)  │ (None, 750, 32)   │     15,392 │ re_lu_34[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 750, 32)   │        128 │ conv1d_35[0][0]   │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_35 (ReLU)     │ (None, 750, 32)   │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_17    │ (None, 187, 32)   │          0 │ re_lu_35[0][0]    │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_21          │ (None, 187, 32)   │          0 │ max_pooling1d_17… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_36 (Conv1D)  │ (None, 187, 64)   │     18,496 │ dropout_21[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 187, 64)   │        256 │ conv1d_36[0][0]   │
│ (BatchNormalizatio… │                   │            │                 

 Total params: 135,765 (530.33 KB)

 Trainable params: 135,061 (527.58 KB)

 Non-trainable params: 704 (2.75 KB)

In [ ]:
model.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=0.001, weight_decay=0.0001),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy'],
)

# Basically just want it to run until nothing is changing
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=30,
        min_delta=0.0001,
        restore_best_weights=True,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=8,
        min_lr=0.0000001
    )
]

history = model.fit(
    [X_tr, F_tr], y_tr.astype('int32'),
    validation_data=([X_val, F_val], y_val.astype('int32')),
    epochs=500, # Super big so it doesn't stop too early
    batch_size=32,
    callbacks=callbacks
)

Epoch 1/500
110/110 ━━━━━━━━━━━━━━━━━━━━ 14s 94ms/step - accuracy: 0.2284 - loss: 2.2724 - val_accuracy: 0.4440 - val_loss: 1.5483 - learning_rate: 0.0010
Epoch 2/500
110/110 ━━━━━━━━━━━━━━━━━━━━ 10s 92ms/step - accuracy: 0.3864 - loss: 1.6480 - val_accuracy: 0.5740 - val_loss: 1.2308 - learning_rate: 0.0010
Epoch 3/500
110/110 ━━━━━━━━━━━━━━━━━━━━ 10s 95ms/step - accuracy: 0.4925 - loss: 1.4120 - val_accuracy: 0.5900 - val_loss: 1.1023 - learning_rate: 0.0010
Epoch 4/500
110/110 ━━━━━━━━━━━━━━━━━━━━ 10s 89ms/step - accuracy: 0.5374 - loss: 1.2631 - val_accuracy: 0.6200 - val_loss: 1.0005 - learning_rate: 0.0010
Epoch 5/500
110/110 ━━━━━━━━━━━━━━━━━━━━ 9s 83ms/step - accuracy: 0.5929 - loss: 1.1474 - val_accuracy: 0.6240 - val_loss: 0.9502 - learning_rate: 0.0010
Epoch 6/500
110/110 ━━━━━━━━━━━━━━━━━━━━ 9s 86ms/step - accuracy: 0.6095 - loss: 1.0899 - val_accuracy: 0.6100 - val_loss: 0.9469 - learning_rate: 0.0010
Epoch 7/500
110/110 ━━━━━━━━━━━━━━━━━━━━ 9s 83ms/step - accuracy: 0.6138

In [ ]:
logits = model.predict([data_test, F_test])

probs = tf.nn.softmax(logits).numpy()

test_pred = pd.DataFrame(probs, columns=[0, 1, 2, 3, 4])
test_pred.to_pickle('../../../test_pred_real_new.xz')

32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step


## Submitting Your Model

After training your classifier, run it on the test data to generate your predictions. Each class for a test sample should have an associated probability (between 0 and 1). Below are the parameters for the prediction format and export:

- Your predictions should be in a pandas dataframe with 5 columns (classes) and 1000 rows (samples). Note that your predictions must follow the original test sample order (0.xz, 1.xz, 2.xz, ...). You only need to worry about this if you shuffled the test samples or stored the samples in an unordered data structure (dictionaries and sets). If this is the case, you should 1) add a separate column in your pandas dataframe with the file number for each sample; 2) sort the dataframe using this column; and 3) drop the column. These steps have been noted in the code below.
- The predictions dataframe should be exported as an .xz file using dataframe.to_pickle() followed by files.download().

Example code of the prediction format and export is presented in the cell block below.

Your model will be evaluated on Area Under the ROC Curve (ROCAUC), Matthews Correlation Coefficient (MCC) and creativity.

If you are finished early, consider trying other ML algorithms and/or implementing multiple feature extraction methods. You can also help other groups if you finish early.

## How Your Model Will Be Evaluated

- **Area Under the ROC Curve (AUCROC)**: The receiver operating characteristic (ROC) curve plots the true positive rate (sensitivity/recall) against the false positive rate (fall-out) at many decision threshold settings. The area under the curve (AUC) measures discrimination, the classifier's ability to correctly identify samples from the "positive" and "negative" cases. Intuitively, AUC is the probability that a randomly chosen "positive" sample will be labeled as "more positive" than a randomly chosen "negative" sample. In the case of a multi-class ROC curve, each class is considered separately before taking the weighted average of all the class results. Simply put, the class under consideration is labeled as "positive" while all other classes are labeled as "negative." Below is the multi-class ROC curve for the example classifier. The AUCROC score should be between 0 and 1, in which 0.5 is random classification and 1 is perfect classification.

<img src="https://github.com/BeaverWorksMedlytics2020/Data_Public/blob/master/Images/Week2/MultiClassRocCurve_exampleClassifier.png?raw=true" width="600" height="500">

- **Matthews Correlation Coefficient (MCC)**: The MCC measures the quality of binary classifications, irrespective of the class sizes. Importantly, it is typically regarded as a balanced measure since it considers all values in the 2x2 contingency table (TP, FP, TN, FN). For this challenge, the binary classes will be "Arousal" (Arousal) and "Nonarousal" (NREM1, NREM2, NREM3, REM). The MCC score should be between -1 and 1, in which 0 is random classification and 1 is perfect classification.

 ![alt text](https://wikimedia.org/api/rest_v1/media/math/render/svg/5caa90fc15105b74b59a30bbc9cc2e5bd43a13b7)

Using these metrics, the example classifier has the following scores on test data:
- AUCROC: 0.727
- MCC: 0.163
- Creativity: ( ͡° ͜ʖ ͡°)

Below is the code used to calculate the AUCROC and MCC metrics when evaluating your classifier.